**Lab type:** prompt  
**Course:** DS104 — Statistics for Data Science  
**Lesson:** Directing AI for Statistical Analysis  
**Task:** Compare weak and strong prompts for three analysis scenarios, then apply the ten-point audit protocol to a flawed AI-generated report.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
n = 1500

department = np.random.choice(
    ['Engineering', 'Sales', 'Support'], size=n, p=[0.40, 0.30, 0.30]
)
manager_rating = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.10, 0.20, 0.40, 0.25])
workload_score = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.10, 0.20, 0.35, 0.25, 0.10])
tenure_years = np.random.exponential(scale=3.5, size=n).clip(0.1, 25).round(1)

raw = 1.5 + 0.5 * manager_rating - 0.25 * workload_score + np.random.normal(0, 1.2, n)
satisfaction_score = np.round(raw).clip(1, 5).astype(int)

left_prob = (0.05 + 0.08 * (5 - satisfaction_score) / 4).clip(0.02, 0.30)
left_company = np.random.binomial(1, left_prob)

df = pd.DataFrame({
    'department': department,
    'satisfaction_score': satisfaction_score,
    'manager_rating': manager_rating,
    'workload_score': workload_score,
    'tenure_years': tenure_years,
    'left_company': left_company,
})

print(f'Dataset: {len(df):,} employees across 3 departments')
print()
print('Variable types:')
print('  satisfaction_score  ordinal 1-5 Likert')
print('  manager_rating      ordinal 1-5 Likert')
print('  workload_score      ordinal 1-5 Likert')
print('  tenure_years        continuous, right-skewed')
print('  left_company        binary (1 = left within 12 months)')
print('  department          categorical')
print()
print(df[['satisfaction_score', 'manager_rating', 'workload_score',
          'tenure_years', 'left_company']].describe().round(2))

## How this lab works

Each task presents a real analysis scenario. You will:

1. Read what a **weak prompt** produced — run the code and examine the output.
2. Write your own **improved prompt** in the designated cell.
3. Compare it to the **strong prompt**, then run the corrected analysis.
4. Apply a **review checklist** to identify which failure mode each weak output triggered.

Task 3 presents a complete AI-generated analysis for you to audit using the ten-point protocol from the lesson.

The goal is not to memorise prompt templates — it is to recognise *why* each failure mode occurs and what a prompt must contain to prevent it.

## Task 1: Correlation analysis with mixed variable types

**Scenario:** Your team wants to know which factors are most strongly associated with employee satisfaction. You send the AI this prompt:

> *"Analyse which factors are most strongly correlated with satisfaction_score."*

Below is the code the AI produced. Run it and examine the output.

In [ ]:
# --- WEAK PROMPT OUTPUT ---
# Prompt: "Analyse which factors are most strongly correlated with satisfaction_score."

numeric_cols = ['manager_rating', 'workload_score', 'tenure_years', 'left_company']
corr = df[numeric_cols + ['satisfaction_score']].corr()['satisfaction_score'].drop('satisfaction_score')

print('Pearson correlation with satisfaction_score:')
print(corr.sort_values(ascending=False).round(3))
print()
print('Conclusion: manager_rating is the strongest predictor of satisfaction.')
print('workload_score and left_company also show meaningful correlations.')

**What did the AI do wrong?**

The AI applied Pearson correlation to all numeric variables — including the three ordinal (Likert) variables. Pearson assumes the relationship is linear and that equal intervals exist between values (1→2 is the same distance as 4→5). Neither holds for a 1–5 rating scale.

A result like r = 0.45 is mathematically computable on ordinal data, but the coefficient's interpretation is incorrect. Spearman rank correlation is the appropriate choice for ordinal variables.

*(Identify the failure mode before continuing.)*

In [ ]:
# --- YOUR PROMPT ---
# Write a stronger prompt as a comment below.
# What should you ask the AI to do before running any correlation analysis?

# Your prompt:
#

**Strong prompt:**

> *Before running any correlation analysis, identify the type of each variable (continuous, ordinal, or categorical). For ordinal variables, use Spearman rank correlation instead of Pearson and explain why. Report the correlation coefficient and p-value for each variable.*

In [ ]:
# --- STRONG PROMPT OUTPUT ---
# Prompt: "Before running any correlation analysis, identify the type of each
#          variable. For ordinal variables, use Spearman rank correlation instead
#          of Pearson and explain why. Report r and p for each."

print('Variable type check:')
print('  satisfaction_score  ordinal (1-5 Likert) — target variable')
print('  manager_rating      ordinal (1-5 Likert) → Spearman')
print('  workload_score      ordinal (1-5 Likert) → Spearman')
print('  tenure_years        continuous            → Spearman (conservative; Pearson also valid)')
print('  left_company        binary                → Spearman')
print()
print('Spearman rank correlations with satisfaction_score:')
for col in ['manager_rating', 'workload_score', 'tenure_years', 'left_company']:
    rho, p = stats.spearmanr(df['satisfaction_score'], df[col])
    print(f'  {col:<22} rho = {rho:+.3f}   p = {p:.4f}')

print()
print('Pearson vs Spearman comparison for ordinal predictors:')
for col in ['manager_rating', 'workload_score']:
    r_pearson = df[['satisfaction_score', col]].corr().loc['satisfaction_score', col]
    r_spearman, _ = stats.spearmanr(df['satisfaction_score'], df[col])
    print(f'  {col:<22} Pearson r = {r_pearson:.3f}   Spearman rho = {r_spearman:.3f}')

**Task 1 review checklist**

| Criterion | Weak prompt | Strong prompt |
|-----------|:-----------:|:-------------:|
| Variable type identified before test selection | | |
| Correct correlation method for ordinal data | | |
| Method choice explained | | |
| p-value reported | | |

1. Which failure mode from the lesson does the weak prompt trigger?

*(Write your answer here.)*

2. Do the Pearson and Spearman results produce a different rank order of "most important factors"? What does that tell you about the risk of using the wrong method?

*(Write your answer here.)*

## Task 2: Group comparison with large n

**Scenario:** HR wants to know whether satisfaction differs across departments. You send the AI:

> *"Is there a statistically significant difference in satisfaction scores across the three departments?"*

Below is the code the AI produced. Run it and examine the conclusion.

In [ ]:
# --- WEAK PROMPT OUTPUT ---
# Prompt: "Is there a statistically significant difference in satisfaction scores
#          across the three departments?"

eng    = df[df.department == 'Engineering']['satisfaction_score']
sales  = df[df.department == 'Sales']['satisfaction_score']
support = df[df.department == 'Support']['satisfaction_score']

h_stat, p_value = stats.kruskal(eng, sales, support)

print('Kruskal-Wallis test:')
print(f'  H = {h_stat:.3f},  p = {p_value:.6f}')
print()
if p_value < 0.001:
    print('Result: Highly significant difference in satisfaction across departments (p < 0.001).')
    print('HR should investigate the causes of these department differences immediately.')

**What did the AI do wrong?**

The AI correctly chose Kruskal-Wallis for ordinal data. The problem is what it did with the result: it declared the difference "highly significant" and escalated to HR based on p < 0.001 alone.

With n = 1,500, even trivially small differences produce very small p-values. p < 0.001 tells you the difference is unlikely to be zero — it says nothing about whether the difference is large enough to matter operationally.

*(Write a prompt that forces the AI to report effect size and frame the business question.)*

In [ ]:
# --- YOUR PROMPT ---
# Write your stronger prompt here as a comment.
# What should you add to get a complete, actionable analysis?

# Your prompt:
#

**Strong prompt:**

> *Run the appropriate test to compare satisfaction scores across the three departments. First verify the test is appropriate for ordinal data. Report: (1) the test statistic and p-value; (2) effect size (epsilon-squared for Kruskal-Wallis); (3) the median for each group; (4) a sentence explicitly distinguishing whether the result is statistically significant, practically significant, or both.*

In [ ]:
# --- STRONG PROMPT OUTPUT ---

eng    = df[df.department == 'Engineering']['satisfaction_score']
sales  = df[df.department == 'Sales']['satisfaction_score']
support = df[df.department == 'Support']['satisfaction_score']

h_stat, p_value = stats.kruskal(eng, sales, support)

# Epsilon-squared: effect size for Kruskal-Wallis
# epsilon^2 = (H - k + 1) / (n - k), where k = number of groups
k = 3
epsilon_sq = (h_stat - k + 1) / (n - k)

print('Variable type: ordinal 1-5 → Kruskal-Wallis appropriate')
print()
print(f'Kruskal-Wallis: H = {h_stat:.3f},  p = {p_value:.6f}')
print(f'Epsilon-squared: {epsilon_sq:.4f}')
print()
print('Group medians:')
for dept, group in [('Engineering', eng), ('Sales', sales), ('Support', support)]:
    print(f'  {dept:<15} median = {group.median():.1f},  mean = {group.mean():.2f},  n = {len(group):,}')
print()

if epsilon_sq < 0.01:
    effect_label = 'negligible'
elif epsilon_sq < 0.06:
    effect_label = 'small'
elif epsilon_sq < 0.14:
    effect_label = 'medium'
else:
    effect_label = 'large'

print(f'Effect size interpretation: {effect_label}')
print(f'  (thresholds: small >= 0.01, medium >= 0.06, large >= 0.14)')
print()
print(f'Interpretation: The test is statistically significant (p < 0.001), but the effect')
print(f'is {effect_label} (epsilon^2 = {epsilon_sq:.4f}). With n = {n:,}, small real differences')
print(f'produce highly significant p-values. Whether this warrants HR action is a business')
print(f'judgment — it depends on whether a difference of this magnitude matters operationally.')

**Task 2 review checklist**

| Criterion | Weak prompt | Strong prompt |
|-----------|:-----------:|:-------------:|
| Correct test for ordinal data | | |
| Effect size reported | | |
| Group-level summaries shown | | |
| Statistical vs practical significance distinguished | | |

1. Which failure mode from the lesson does the weak prompt trigger?

*(Write your answer here.)*

2. Look at the epsilon-squared value from the strong output. Would you recommend HR escalate based on this result? What additional context would you need to make that judgment?

*(Write your answer here.)*

## Task 3: Audit a complete AI-generated analysis

Below is a full statistical analysis produced by an AI tool in response to:

> *"Identify the key predictors of employee turnover. Find which variables are significantly associated with left_company."*

Read the code and its output carefully. There are **three errors** embedded in the analysis and its conclusions. Your job is to find them using the ten-point audit protocol from the lesson.

In [ ]:
# --- AI-GENERATED ANALYSIS (unedited) ---
# Prompt: "Identify the key predictors of employee turnover.
#          Find which variables are significantly associated with left_company."

print('=== EMPLOYEE TURNOVER PREDICTOR ANALYSIS ===')
print()

# Step 1: Correlation screening — all numeric features vs left_company
features = ['satisfaction_score', 'manager_rating', 'workload_score', 'tenure_years']
print('Pearson correlations with left_company:')
for feat in features:
    r, p = stats.pearsonr(df[feat], df['left_company'])
    sig = ' **' if p < 0.01 else (' *' if p < 0.05 else '')
    print(f'  {feat:<22} r = {r:+.3f}   p = {p:.4f}{sig}')

print()
print('Significant predictors (p < 0.05): satisfaction_score, manager_rating, workload_score')
print()

# Step 2: Confirm with a direct group comparison
leavers = df[df.left_company == 1]['satisfaction_score']
stayers = df[df.left_company == 0]['satisfaction_score']

t_stat, p_value = stats.ttest_ind(leavers, stayers)

print(f'Satisfaction score — leavers: mean = {leavers.mean():.2f},  stayers: mean = {stayers.mean():.2f}')
print(f't-test: t = {t_stat:.3f},  p = {p_value:.6f}')
print()
print('This result is highly significant (p < 0.001).')
print('There is a 99.9% probability that low satisfaction causes higher turnover.')
print('HR should prioritise satisfaction improvement to directly reduce attrition.')

## Apply the audit protocol

Use the ten-point checklist below to evaluate the AI output. For each item, mark **✓** (passed), **✗** (failed), or **—** (not applicable to this analysis). Where the AI failed, write one sentence describing the error and how to fix it.

```
□ Was the variable type checked? (continuous vs ordinal vs categorical)
□ Was the correct test selected for the variable type and comparison structure?
□ Were test assumptions verified? (normality, equal variance, cell counts)
□ If normality failed, was a non-parametric alternative used?
□ Are effect sizes reported alongside p-values?
□ If multiple tests were run, was a multiple comparison correction applied?
□ Are confidence intervals reported for all point estimates?
□ Does the written interpretation correctly distinguish statistical from practical significance?
□ Does the commentary avoid causal language where only correlational analysis was done?
□ Were distributions visualised before any modelling or test decisions were made?
```

*(Fill in your audit here — mark each item and explain failures.)*

---

**Identified errors (there are exactly three):**

1. *(describe error and which checklist item it violates)*
2. *(describe error and which checklist item it violates)*
3. *(describe error and which checklist item it violates)*

---

**Fix it:** In the cell below, write a corrected version of the Step 2 group comparison. Apply the correct test, verify assumptions, and rewrite the conclusion without the errors you identified.

In [ ]:
# --- YOUR CORRECTED ANALYSIS ---
# Fix the three errors from the audit above.
# Hints: check normality first, report effect size, rewrite the conclusion.

leavers = df[df.left_company == 1]['satisfaction_score']
stayers = df[df.left_company == 0]['satisfaction_score']

# Your corrected code here:


## Summary

> **Final question:** In one sentence each, state the three lessons from this lab.

1. 
2. 
3. 